# Day 3: Gradient Boosting - XGBoost

## Overview

XGBoost is the dominant algorithm in ML competitions. Sequential tree ensembles that fit residuals, with built-in regularisation and early stopping. Master the hyperparameters and evaluation strategies.

## Learning Objectives

- Understand gradient boosting: sequential error correction
- Learn XGBoost-specific features (early stopping, regularisation)
- Tune learning_rate, max_depth, n_estimators
- Extract feature importance and monitor training

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
import xgboost as xgb

np.random.seed(42)
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Understanding Gradient Boosting

In [ ]:
# Create synthetic classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=8,
                             n_redundant=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train XGBoost with different numbers of rounds
rounds_list = [10, 50, 100, 200, 500]
train_scores = []
test_scores = []

for rounds in rounds_list:
    clf = xgb.XGBClassifier(n_estimators=rounds, max_depth=5, learning_rate=0.1,
                              random_state=42, verbosity=0)
    clf.fit(X_train, y_train)
    train_pred = clf.predict(X_train)
    test_pred = clf.predict(X_test)
    train_scores.append(accuracy_score(y_train, train_pred))
    test_scores.append(accuracy_score(y_test, test_pred))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rounds_list, train_scores, marker='o', label='Train', linewidth=2)
ax.plot(rounds_list, test_scores, marker='s', label='Test', linewidth=2)
ax.set_xlabel('Number of Boosting Rounds')
ax.set_ylabel('Accuracy')
ax.set_title('XGBoost: Sequential Error Correction')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Train accuracy progression: {[f'{s:.3f}' for s in train_scores]}")
print(f"Test accuracy progression: {[f'{s:.3f}' for s in test_scores]}")

## 2. Early Stopping

In [ ]:
# Use eval_set for early stopping
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Train with early stopping
clf_early = xgb.XGBClassifier(n_estimators=1000, max_depth=5, learning_rate=0.1,
                                random_state=42, verbosity=0)
clf_early.fit(X_train_split, y_train_split,
               eval_set=[(X_val, y_val)],
               early_stopping_rounds=20,
               verbose=False)

# Get results
best_rounds = clf_early.best_iteration + 1
print(f"Early stopping at round: {best_rounds}")
print(f"Test accuracy (early stopped): {accuracy_score(y_test, clf_early.predict(X_test)):.4f}")

# Compare to no early stopping
clf_no_early = xgb.XGBClassifier(n_estimators=best_rounds, max_depth=5, learning_rate=0.1,
                                   random_state=42, verbosity=0)
clf_no_early.fit(X_train, y_train)
print(f"Test accuracy (full train): {accuracy_score(y_test, clf_no_early.predict(X_test)):.4f}")

## 3. Learning Rate and Tree Depth

In [ ]:
# Test learning rates
learning_rates = [0.01, 0.05, 0.1, 0.2, 0.3]
lr_scores = []

for lr in learning_rates:
    clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=lr,
                              random_state=42, verbosity=0)
    clf.fit(X_train, y_train)
    lr_scores.append(accuracy_score(y_test, clf.predict(X_test)))

# Test max_depth values
depths = [3, 5, 7, 10, 15]
depth_scores = []

for depth in depths:
    clf = xgb.XGBClassifier(n_estimators=100, max_depth=depth, learning_rate=0.1,
                              random_state=42, verbosity=0)
    clf.fit(X_train, y_train)
    depth_scores.append(accuracy_score(y_test, clf.predict(X_test)))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(learning_rates, lr_scores, marker='o', linewidth=2, markersize=8)
ax1.set_xlabel('Learning Rate (eta)')
ax1.set_ylabel('Test Accuracy')
ax1.set_title('Effect of Learning Rate')
ax1.grid(alpha=0.3)

ax2.plot(depths, depth_scores, marker='o', linewidth=2, markersize=8)
ax2.set_xlabel('max_depth')
ax2.set_ylabel('Test Accuracy')
ax2.set_title('Effect of Tree Depth')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best LR: {learning_rates[np.argmax(lr_scores)]} → {max(lr_scores):.4f}")
print(f"Best Depth: {depths[np.argmax(depth_scores)]} → {max(depth_scores):.4f}")

## 4. Feature Importance

In [ ]:
# Train final model
clf_final = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                random_state=42)
clf_final.fit(X_train, y_train)

# Get feature importance (gain = improvement in accuracy)
importance_gain = clf_final.get_booster().get_score(importance_type='gain')
importance_freq = clf_final.get_booster().get_score(importance_type='frequency')

# Convert to DataFrame for plotting
importance_df = pd.DataFrame({
    'Feature': [f'f{i}' for i in range(X_train.shape[1])],
    'Gain': [importance_gain.get(f'f{i}', 0) for i in range(X_train.shape[1])],
    'Frequency': [importance_freq.get(f'f{i}', 0) for i in range(X_train.shape[1])]
}).sort_values('Gain', ascending=True)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(importance_df['Feature'], importance_df['Gain'])
ax1.set_xlabel('Gain (Improvement in Accuracy)')
ax1.set_title('XGBoost Feature Importance (Gain)')

ax2.barh(importance_df['Feature'], importance_df['Frequency'])
ax2.set_xlabel('Frequency (Usage Count)')
ax2.set_title('XGBoost Feature Importance (Frequency)')

plt.tight_layout()
plt.show()

print(importance_df)

## 5. Handling Missing Values

In [ ]:
# Create data with missing values
X_with_missing = X_train.copy()
# Introduce 20% missing values
mask = np.random.random(X_with_missing.shape) < 0.2
X_with_missing[mask] = np.nan

# XGBoost handles missing natively
clf_missing = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                  random_state=42)
clf_missing.fit(X_with_missing, y_train)

# Predict on test set (no missing)
score_missing = accuracy_score(y_test, clf_missing.predict(X_test))

print(f"Test accuracy with missing data handling: {score_missing:.4f}")
print(f"This is why XGBoost doesn't need explicit imputation!")

## Exercises

1. Compare XGBoost vs Random Forest on the same dataset. Which is faster? Which is more accurate?
2. Use GridSearchCV to find optimal learning_rate and max_depth.
3. Plot training loss vs validation loss to visualize overfitting.

## Solutions

Key takeaways:
- Boosting fits residuals sequentially for continued improvement
- Early stopping prevents overfitting and saves time
- Low learning_rate + high n_estimators often beats high learning_rate
- XGBoost handles missing values automatically